# Data consolidation

* The goal is to consolidate all of the session, race, and pit data into one large dataset

In [51]:
import pandas as pd
import numpy as np 
import os
import sys
from sklearn.preprocessing import OneHotEncoder
from functools import reduce


sys.path.append(os.path.abspath(".."))
from Helpers.utils import get_supabase, get_supabase_df

In [52]:
# initizalize supabase client
supabase = get_supabase()

# get tables
sessions_df = get_supabase_df("sessions")
driver_df = get_supabase_df("drivers")
weather_df = get_supabase_df("weather")
pit_df = get_supabase_df("pits")
results_df = get_supabase_df("results")
lap_df = get_supabase_df("laps")

## Sessions Table Processing

In [53]:
# add date column and remove date_start and date_end columns
sessions_df['date_start'] = pd.to_datetime(sessions_df['date_start'])
sessions_df['date'] = sessions_df['date_start'].dt.date
sessions_df["date"] = pd.to_datetime(sessions_df["date"])

# date columns 
sessions_df["month"] = sessions_df["date"].dt.month.astype("int64")
sessions_df["day"] = sessions_df["date"].dt.day.astype("int64")

# reorder columns
sessions_df = sessions_df[['meeting_key', 'session_key', 'session_type', 'session_name', 'country_key', "circuit_key", "year", "month", "day"]]

print(sessions_df["session_name"].value_counts())
print(sessions_df["session_type"].value_counts())

# remove sprints
sessions_df.head()

session_name
Practice 1           44
Qualifying           43
Race                 43
Practice 2           34
Practice 3           34
Sprint Qualifying    10
Sprint               10
Day 1                 1
Day 2                 1
Day 3                 1
Name: count, dtype: int64
session_type
Practice      115
Qualifying     53
Race           53
Name: count, dtype: int64


,meeting_key,session_key,session_type,session_name,country_key,circuit_key,year,month,day
0,1231,9484,Qualifying,Qualifying,5,10,2024,3,23
1,1231,9488,Race,Race,5,10,2024,3,24
2,1232,9489,Practice,Practice 1,4,46,2024,4,5
3,1232,9490,Practice,Practice 2,4,46,2024,4,5
4,1232,9491,Practice,Practice 3,4,46,2024,4,6


In [54]:
# encoding categorical variables
encoder = OneHotEncoder(drop=None, sparse_output=False) # sparse_output=False for dense array

# Fit and transform the categorical column
encoded_features = encoder.fit_transform(sessions_df[['session_type']]).astype("int64")

# Create a DataFrame from the encoded features
processed_sessions = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(['session_type']))

# Concatenate with the original DataFrame (excluding the original categorical column)
processed_sessions = pd.concat([sessions_df.drop('session_type', axis=1), processed_sessions], axis=1)

# handle numeric columns
numeric_cols = ['meeting_key', 'session_key', 'country_key', 'circuit_key', 
                'year', 'month', 'day', 
                'session_type_Practice', 'session_type_Qualifying', 'session_type_Race']

# Convert to pandas nullable Int64 dtype
processed_sessions[numeric_cols] = processed_sessions[numeric_cols].astype('Int64')

processed_sessions.head()

,meeting_key,session_key,session_name,country_key,circuit_key,year,month,day,session_type_Practice,session_type_Qualifying,session_type_Race
0,1231,9484,Qualifying,5,10,2024,3,23,0,1,0
1,1231,9488,Race,5,10,2024,3,24,0,0,1
2,1232,9489,Practice 1,4,46,2024,4,5,1,0,0
3,1232,9490,Practice 2,4,46,2024,4,5,1,0,0
4,1232,9491,Practice 3,4,46,2024,4,6,1,0,0


## Drivers Table Processing

In [55]:
# get drivers
driver_df.drop(columns = ["country_code","team_name", "name_acronym"], inplace=True, errors="ignore")
processed_drivers = driver_df.copy()
processed_drivers.head()

,meeting_key,session_key,driver_number
0,1233,9672,4
1,1233,9672,10
2,1233,9672,11
3,1233,9672,14
4,1233,9672,16


## Weather Table Processing

In [56]:
# change weather date
weather_df["date"] = pd.to_datetime(weather_df["date"])

# separate dates into int columns
weather_df["year"] = weather_df["date"].dt.year.astype("int64")
weather_df["month"] = weather_df["date"].dt.month.astype("int64")
weather_df["day"] = weather_df["date"].dt.day.astype("int64")

# columns for rain status
weather_df["rainfall_status"] = np.where(weather_df["rainfall"] > 0, True, False)

# reorder colummns
processed_weather = weather_df[["session_key","meeting_key",'pressure', 'rainfall_status',
       'track_temperature', 'wind_speed', 'wind_direction',
       'air_temperature', "year", "month", "day"]]
processed_weather.head()

,session_key,meeting_key,pressure,rainfall_status,track_temperature,wind_speed,wind_direction,air_temperature,year,month,day
0,9663,1233,1009.332927,False,39.468293,1.353659,146.963415,23.142683,2024,4,19
1,9567,1242,964.786747,True,38.386747,1.551807,156.012048,22.012048,2024,7,26
2,9599,1246,1005.607500,False,37.115000,2.448750,187.900000,31.350000,2024,9,20
3,10015,1258,1006.662821,False,46.964103,1.643590,285.294872,28.053846,2025,4,18
4,9981,1260,1007.209877,False,34.912346,2.328395,128.012346,19.676543,2025,5,16


## Pit Table Processing

In [57]:
# get median pit times instead of all of the pits
pit_df = pd.DataFrame(pit_df.groupby(["meeting_key","session_key","date","driver_number"])["pit_duration"].median().reset_index())
pit_df.rename(columns={"pit_duration":"median_pit_durations"}, inplace=True)

# change weather date
pit_df["date"] = pd.to_datetime(pit_df["date"])

# separate dates into int columns
pit_df["year"] = pit_df["date"].dt.year.astype("int64")
pit_df["month"] = pit_df["date"].dt.month.astype("int64")
pit_df["day"] = pit_df["date"].dt.day.astype("int64")

# keep columns
processed_pits = pit_df[['meeting_key', 'session_key', 'driver_number',
       'median_pit_durations', 'year', 'month', 'day']].copy()
processed_pits.head()

,meeting_key,session_key,driver_number,median_pit_durations,year,month,day
0,1228,9462,1,38.4,2024,2,21
1,1228,9462,2,302.2,2024,2,21
2,1228,9462,3,35.8,2024,2,21
3,1228,9462,4,456.6,2024,2,21
4,1228,9462,10,33.7,2024,2,21


## Laps Table Processing

In [58]:
# column manipulation
lap_df[["lap_duration","duration_sector_1","duration_sector_2","duration_sector_3","st_speed"]] = lap_df[["lap_duration","duration_sector_1","duration_sector_2","duration_sector_3","st_speed"]].astype(float)
lap_df = lap_df.dropna(subset=["duration_sector_1", "duration_sector_2", "duration_sector_3"], how="all")
lap_df['date_start'] = pd.to_datetime(lap_df['date_start'], format='ISO8601',errors="ignore")
lap_df['date_start'] = lap_df['date_start'].dt.date

# group by average sector times
lap_df = (
    lap_df.groupby(["meeting_key", "session_key", "driver_number"], as_index=False)
    .agg(
        avg_duration_sector_1=("duration_sector_1", "mean"),
        avg_duration_sector_2=("duration_sector_2", "mean"),
        avg_duration_sector_3=("duration_sector_3", "mean"),
        best_lap_duration=("lap_duration", "min"),
        median_top_speed=("st_speed", "median")
    )
)

# rename column
lap_df = lap_df.rename(columns={"duration_sector_1":"avg_duration_sector_1","duration_sector_2":"avg_duration_sector_2","duration_sector_3":"avg_duration_sector_3","lap_duration":"best_lap_duration","st_speed":"median_top_speed"})
processed_laps = lap_df.copy()
processed_laps.head()

/var/folders/59/0pv0md114td124t9vtdq3nhc0000gn/T/ipykernel_7274/3229853858.py:4: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  lap_df['date_start'] = pd.to_datetime(lap_df['date_start'], format='ISO8601',errors="ignore")


,meeting_key,session_key,driver_number,avg_duration_sector_1,avg_duration_sector_2,avg_duration_sector_3,best_lap_duration,median_top_speed
0,1228,9462,1,37.231427,45.966622,25.834552,91.344,288.0
1,1228,9462,2,36.030176,50.045333,28.429048,93.882,290.0
2,1228,9462,3,37.303955,46.218627,25.879212,92.599,284.5
3,1228,9462,4,33.037344,44.029528,25.134438,92.484,286.0
4,1228,9462,10,39.444412,45.986033,26.695279,92.805,271.5


## Results Table Processing

In [59]:
results_df.drop(columns=["points","gap_to_leader"], inplace=True, errors="ignore")
results_df.rename(columns={"duration":"best_lap_time", "position":"end_position"}, inplace=True)
results_df["best_lap_time"] = pd.to_numeric(results_df["best_lap_time"], errors='coerce')
processed_results = results_df[['meeting_key', 'session_key', 'driver_number',
       'number_of_laps', 'dnf', 'dns', 'dsq', 'best_lap_time','end_position']].copy()

print(processed_results.shape)
processed_results.head()

(4273, 9)


,meeting_key,session_key,driver_number,number_of_laps,dnf,dns,dsq,best_lap_time,end_position
0,1230,9474,14,28,false,false,false,88.827,1
1,1230,9474,63,24,false,false,false,89.057,2
2,1230,9474,1,27,false,false,false,89.158,3
3,1230,9474,16,25,false,false,false,89.180,4
4,1230,9474,11,27,false,false,false,89.300,5


## Verify Valid Sessions

In [60]:
# after you filter sessions_df to remove sprint races
valid_session_keys = sessions_df['session_key'].unique()
valid_meeting_keys = sessions_df['meeting_key'].unique()

# Filter other processed tables to the same session_key set
processed_results = processed_results[processed_results['session_key'].isin(valid_session_keys)]
processed_laps    = processed_laps[processed_laps['session_key'].isin(valid_session_keys)]
processed_weather = processed_weather[processed_weather['session_key'].isin(valid_session_keys)]
processed_pits    = processed_pits[processed_pits['session_key'].isin(valid_session_keys)]


## Master Table Joining

In [61]:
# merge dfs
master_df = processed_results.merge(processed_sessions, on=['meeting_key','session_key'], how='inner')
master_df = master_df.merge(processed_weather, on=['meeting_key','session_key'], how='left')

# remove duplicate columns
master_df.drop(columns=['year_y', 'month_y', 'day_y'], inplace=True)
master_df.rename(columns={"year_x":"year","month_x":"month","day_x":"day"}, inplace = True)

print(master_df.columns)
print(master_df.shape)
master_df[master_df["session_type_Race"] == 1].tail(30)

Index(['meeting_key', 'session_key', 'driver_number', 'number_of_laps', 'dnf',
       'dns', 'dsq', 'best_lap_time', 'end_position', 'session_name',
       'country_key', 'circuit_key', 'year', 'month', 'day',
       'session_type_Practice', 'session_type_Qualifying', 'session_type_Race',
       'pressure', 'rainfall_status', 'track_temperature', 'wind_speed',
       'wind_direction', 'air_temperature'],
      dtype='object')
(4273, 24)


,meeting_key,session_key,driver_number,number_of_laps,dnf,dns,dsq,best_lap_time,end_position,session_name,...,day,session_type_Practice,session_type_Qualifying,session_type_Race,pressure,rainfall_status,track_temperature,wind_speed,wind_direction,air_temperature
4163,1269,9904,5,51,false,false,false,5674.014,11,Race,...,21,0,0,1,1021.116026,False,26.107051,1.507051,160.762821,20.551923
4164,1269,9904,87,51,false,false,false,5674.670,12,Race,...,21,0,0,1,1021.116026,False,26.107051,1.507051,160.762821,20.551923
4165,1269,9904,23,51,false,false,false,5679.278,13,Race,...,21,0,0,1,1021.116026,False,26.107051,1.507051,160.762821,20.551923
4166,1269,9904,31,51,false,false,false,5683.988,14,Race,...,21,0,0,1,1021.116026,False,26.107051,1.507051,160.762821,20.551923
4167,1269,9904,14,51,false,false,false,5685.115,15,Race,...,21,0,0,1,1021.116026,False,26.107051,1.507051,160.762821,20.551923
4168,1269,9904,27,51,false,false,false,5686.645,16,Race,...,21,0,0,1,1021.116026,False,26.107051,1.507051,160.762821,20.551923
4169,1269,9904,18,51,false,false,false,5702.800,17,Race,...,21,0,0,1,1021.116026,False,26.107051,1.507051,160.762821,20.551923
4170,1269,9904,10,50,false,false,false,0.000,18,Race,...,21,0,0,1,1021.116026,False,26.107051,1.507051,160.762821,20.551923
4171,1269,9904,43,50,false,false,false,0.000,19,Race,...,21,0,0,1,1021.116026,False,26.107051,1.507051,160.762821,20.551923
4172,1269,9904,81,0,true,false,false,0.000,0,Race,...,21,0,0,1,1021.116026,False,26.107051,1.507051,160.762821,20.551923


In [62]:
# handle corrupt data where drivers have no lap time but have ending position
bool_map = {'true': True, 'false': False, True: True, False: False}

master_df['dnf'] = master_df['dnf'].map(bool_map)
master_df['dns'] = master_df['dns'].map(bool_map)
master_df['dsq'] = master_df['dsq'].map(bool_map)

invalid_mask = (
    (master_df['best_lap_time'] == 0) &
    (master_df['end_position'] > 0) &
    (~master_df['dnf']) & 
    (~master_df['dns']) & 
    (~master_df['dsq'])
)
master_df = master_df[~invalid_mask]
print(master_df.columns)

Index(['meeting_key', 'session_key', 'driver_number', 'number_of_laps', 'dnf',
       'dns', 'dsq', 'best_lap_time', 'end_position', 'session_name',
       'country_key', 'circuit_key', 'year', 'month', 'day',
       'session_type_Practice', 'session_type_Qualifying', 'session_type_Race',
       'pressure', 'rainfall_status', 'track_temperature', 'wind_speed',
       'wind_direction', 'air_temperature'],
      dtype='object')


### Merging Quali and Practice Data together and then combining it with actual Race Results

In [63]:
# convert get best lap times in quali session
sector_laps = master_df.merge(processed_laps, how = "left" ,on = ["meeting_key","session_key","driver_number"])
quali_laps = sector_laps[sector_laps["session_type_Qualifying"] == 1]
quali_laps.columns
quali_laps = quali_laps[['meeting_key', 'driver_number','best_lap_duration', 'median_top_speed', 'end_position']]
quali_laps.rename(columns={"best_lap_duration":"quali_best_lap","median_top_speed":"quali_median_ts","end_position":"race_start_position"}, inplace=True)
print(f"shape before drop: {quali_laps.shape}")
quali_laps = quali_laps.dropna(subset=['meeting_key','driver_number','quali_best_lap'])
print(f"shape after drop: {quali_laps.shape}")
quali_laps.head()

shape before drop: (998, 5)
shape after drop: (996, 5)


,meeting_key,driver_number,quali_best_lap,quali_median_ts,race_start_position
22,1233,44,95.573,262.5,18
23,1233,22,95.746,289.0,19
24,1233,2,96.358,293.0,20
81,1243,4,69.673,328.0,1
82,1243,1,70.029,327.0,2


In [64]:
quali_laps = quali_laps.drop_duplicates(subset=['meeting_key', 'driver_number'], keep='last')
print(f"Quali laps after deduplication: {quali_laps.shape}")

Quali laps after deduplication: (836, 5)


In [66]:
# Filter for practice sessions
practice_laps = sector_laps[sector_laps["session_type_Practice"] == 1].copy()

# Normalize session names more robustly
practice_laps["session_name"] = practice_laps["session_name"].replace({
    "Day 1": "Practice 1",
    "Day 2": "Practice 2", 
    "Day 3": "Practice 3",
    "Practice": "Practice 1",
    "Free Practice 1": "Practice 1",
    "Free Practice 2": "Practice 2",
    "Free Practice 3": "Practice 3"
})

# Compute average duration
practice_laps["avg_lap_duration"] = (
    practice_laps["avg_duration_sector_1"] +
    practice_laps["avg_duration_sector_2"] +
    practice_laps["avg_duration_sector_3"]
)

# Keep relevant columns
practice_laps = practice_laps[[
    "meeting_key", "driver_number", "session_name",
    "best_lap_duration", "avg_lap_duration", "median_top_speed"
]].rename(columns={
    "best_lap_duration": "practice_best_lap",
    "median_top_speed": "practice_median_ts"
})

# AGGREGATE BEFORE PIVOT to handle multiple sessions of same type
practice_laps = practice_laps.groupby(
    ['meeting_key', 'driver_number', 'session_name']
).agg({
    'practice_best_lap': 'min',
    'practice_median_ts': 'median'
}).reset_index()

# Now pivot with confidence
practice_laps = practice_laps.pivot(
    index=["meeting_key", "driver_number"],
    columns="session_name", 
    values="practice_best_lap"
).reset_index()

# Ensure all 3 practice columns exist
for col in ["Practice 1", "Practice 2", "Practice 3"]:
    if col not in practice_laps.columns:
        practice_laps[col] = np.nan

# Rename columns
practice_laps.columns = [
    f"{col}_best_lap" if col not in ["meeting_key", "driver_number"] else col
    for col in practice_laps.columns
]

# Flag sprint weekends (FP2 & FP3 missing entirely)
meeting_counts = practice_laps.groupby("meeting_key")[["Practice 2_best_lap", "Practice 3_best_lap"]].apply(
    lambda df: df.notna().any().to_dict()
).apply(pd.Series)

sprint_meetings = meeting_counts[
    (meeting_counts["Practice 2_best_lap"] == False) &
    (meeting_counts["Practice 3_best_lap"] == False)
].index

practice_laps["is_sprint_weekend"] = practice_laps["meeting_key"].isin(sprint_meetings)
practice_laps.tail(30)


,meeting_key,driver_number,Practice 1_best_lap,Practice 2_best_lap,Practice 3_best_lap,is_sprint_weekend
889,1270,30,92.461,92.645,93.628,False
890,1270,31,92.128,91.298,90.784,False
891,1270,43,93.324,93.139,91.047,False
892,1270,44,91.480,91.491,90.559,False
893,1270,55,91.812,91.299,90.392,False
894,1270,63,92.139,93.231,90.197,False
895,1270,81,91.481,90.714,90.165,False
896,1270,87,92.538,91.711,90.799,False
897,1277,1,87.432,86.314,85.585,False
898,1277,4,86.915,85.816,85.606,False


In [67]:
practice_laps = practice_laps.drop_duplicates(subset=['meeting_key', 'driver_number'], keep='last')
print(f"Practice laps after deduplication: {practice_laps.shape}")


Practice laps after deduplication: (919, 6)


In [68]:
# get race_df
race_df = master_df[master_df["session_type_Race"] == 1]
race_df = race_df[['meeting_key', 'session_key', 'driver_number','dnf', 'dns', 'dsq','end_position','country_key', 'circuit_key', 'year', 'month', 'day','pressure', 'rainfall_status', 'track_temperature', 'wind_speed',
       'wind_direction', 'air_temperature']]
race_df

,meeting_key,session_key,driver_number,dnf,dns,dsq,end_position,country_key,circuit_key,year,month,day,pressure,rainfall_status,track_temperature,wind_speed,wind_direction,air_temperature
20,1234,9506,18,True,False,False,0,19,151,2024,5,4,1016.842683,False,44.826829,3.109756,152.670732,28.698780
21,1234,9506,4,True,False,False,0,19,151,2024,5,4,1016.842683,False,44.826829,3.109756,152.670732,28.698780
25,1233,9673,1,False,False,False,1,53,49,2024,4,21,1011.452121,False,29.792727,1.680000,147.551515,18.639394
26,1233,9673,4,False,False,False,2,53,49,2024,4,21,1011.452121,False,29.792727,1.680000,147.551515,18.639394
27,1233,9673,11,False,False,False,3,53,49,2024,4,21,1011.452121,False,29.792727,1.680000,147.551515,18.639394
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4257,1270,9896,12,False,False,False,5,157,61,2025,10,5,1010.782500,True,33.850625,1.110000,149.706250,29.074375
4258,1270,9896,16,False,False,False,6,157,61,2025,10,5,1010.782500,True,33.850625,1.110000,149.706250,29.074375
4259,1270,9896,14,False,False,False,7,157,61,2025,10,5,1010.782500,True,33.850625,1.110000,149.706250,29.074375
4260,1270,9896,44,False,False,False,8,157,61,2025,10,5,1010.782500,True,33.850625,1.110000,149.706250,29.074375


In [69]:
race_df = race_df.drop_duplicates(subset=['meeting_key', 'driver_number'], keep='last')
print(f"Race data after deduplication: {race_df.shape}")

Race data after deduplication: (651, 18)


In [73]:

# Add validation function
def validate_driver_meeting_uniqueness(df, step_name):
    """Validate that each driver appears only once per meeting"""
    pk_cols = ['meeting_key', 'driver_number']
    duplicates = df[df.duplicated(subset=pk_cols, keep=False)]
    
    if len(duplicates) > 0:
        print(f"⚠️  {step_name}: Found {len(duplicates)} duplicate rows")
        print("Sample duplicates:")
        print(duplicates[pk_cols].head())
        return False
    else:
        print(f"✅ {step_name}: No duplicates found ({len(df)} rows)")
        return True# join practice and qualifying
        
# Join with validation
joined_df = race_df \
    .merge(quali_laps, on=["meeting_key","driver_number"], how='left') \
    .merge(practice_laps, on=["meeting_key","driver_number"], how='left')

print(f"\nPost-join shape: {joined_df.shape}")

# Validate final result
validate_driver_meeting_uniqueness(joined_df, "Final Joined Data")

# If duplicates found, remove them
if joined_df.duplicated(subset=['meeting_key', 'driver_number']).any():
    print("Removing duplicates from final dataset...")
    joined_df = joined_df.drop_duplicates(subset=['meeting_key', 'driver_number'], keep='last')
    print(f"Final shape after deduplication: {joined_df.shape}")

# reorder columns
joined_df = joined_df[['meeting_key', 'driver_number', 'quali_best_lap', 'quali_median_ts',
       'race_start_position',
       'Practice 1_best_lap', 'Practice 2_best_lap', 'Practice 3_best_lap',
       'is_sprint_weekend',
       'country_key', 'circuit_key', 'year', 'month', 'day', 'pressure',
       'rainfall_status', 'track_temperature', 'wind_speed', 'wind_direction',
       'air_temperature', 'end_position']]

joined_df.rename(columns = { 'Practice 1_best_lap' : 'Practice1_best_lap', 'Practice 2_best_lap':'Practice2_best_lap',
       'Practice 3_best_lap':'Practice3_best_lap'}, inplace=True)

# change start position dtype
joined_df["race_start_position"] = joined_df["race_start_position"].astype('Int64')
print(joined_df.columns)
joined_df.tail(30)


Post-join shape: (651, 25)
✅ Final Joined Data: No duplicates found (651 rows)
Index(['meeting_key', 'driver_number', 'quali_best_lap', 'quali_median_ts',
       'race_start_position', 'Practice1_best_lap', 'Practice2_best_lap',
       'Practice3_best_lap', 'is_sprint_weekend', 'country_key', 'circuit_key',
       'year', 'month', 'day', 'pressure', 'rainfall_status',
       'track_temperature', 'wind_speed', 'wind_direction', 'air_temperature',
       'end_position'],
      dtype='object')


,meeting_key,driver_number,quali_best_lap,quali_median_ts,race_start_position,Practice1_best_lap,Practice2_best_lap,Practice3_best_lap,is_sprint_weekend,country_key,...,year,month,day,pressure,rainfall_status,track_temperature,wind_speed,wind_direction,air_temperature,end_position
621,1268,30,80.279,213.0,20,81.201,80.811,80.132,False,13,...,2025,9,7,996.752778,False,43.288194,1.960417,184.381944,26.761111,14
622,1268,14,79.362,212.5,9,81.114,80.645,79.861,False,13,...,2025,9,7,996.752778,False,43.288194,1.960417,184.381944,26.761111,0
623,1268,27,79.498,243.0,12,81.179,80.241,79.737,False,13,...,2025,9,7,996.752778,False,43.288194,1.960417,184.381944,26.761111,0
624,1269,1,101.117,306.0,1,103.790,101.902,101.445,False,30,...,2025,9,21,1021.116026,False,26.107051,1.507051,160.762821,20.551923,1
625,1269,63,101.455,307.0,5,103.257,101.770,101.964,False,30,...,2025,9,21,1021.116026,False,26.107051,1.507051,160.762821,20.551923,2
626,1269,55,101.595,296.5,2,103.859,102.255,102.486,False,30,...,2025,9,21,1021.116026,False,26.107051,1.507051,160.762821,20.551923,3
627,1269,12,101.464,304.5,4,103.985,101.779,101.876,False,30,...,2025,9,21,1021.116026,False,26.107051,1.507051,160.762821,20.551923,4
628,1269,30,101.537,306.0,3,103.903,101.989,102.146,False,30,...,2025,9,21,1021.116026,False,26.107051,1.507051,160.762821,20.551923,5
629,1269,22,101.788,306.0,6,103.738,102.444,102.840,False,30,...,2025,9,21,1021.116026,False,26.107051,1.507051,160.762821,20.551923,6
630,1269,4,101.322,304.0,7,102.704,102.199,101.223,False,30,...,2025,9,21,1021.116026,False,26.107051,1.507051,160.762821,20.551923,7


# Master_Df Supabase Upsertion

In [75]:
# Updated upsertion code with deduplication

# Step 1: Replace all problematic values
joined_df = joined_df.replace([np.inf, -np.inf], None)
joined_df = joined_df.replace({np.nan: None})

# Step 2: For any remaining float columns, ensure they're clean
for col in joined_df.select_dtypes(include=['float64', 'float32']).columns:
    joined_df[col] = joined_df[col].astype('object')
    joined_df[col] = joined_df[col].replace({np.nan: None})
    joined_df[col] = pd.to_numeric(joined_df[col], errors='ignore')

# Step 3: CRITICAL - Remove duplicates before upsertion
print(f"DataFrame shape before deduplication: {joined_df.shape}")

# Check for duplicates on the primary key columns
pk_cols = ['meeting_key', 'driver_number']
duplicates = joined_df[joined_df.duplicated(subset=pk_cols, keep=False)]
print(f"Found {len(duplicates)} duplicate rows on {pk_cols}")

if len(duplicates) > 0:
    print("Sample duplicates:")
    print(duplicates[pk_cols].head(10))
    
    # Remove duplicates, keeping the last occurrence (you can change to 'first' if needed)
    joined_df = joined_df.drop_duplicates(subset=pk_cols, keep='last')
    print(f"DataFrame shape after deduplication: {joined_df.shape}")

# Step 4: Use pandas built-in method that handles NaN better
records = joined_df.to_dict(orient="records")

# Step 5: Manual cleanup of any remaining NaN in records
for record in records:
    for key, value in record.items():
        if pd.isna(value) or (isinstance(value, float) and np.isnan(value)):
            record[key] = None

# Step 6: Upsert using meeting_key + driver_number as the composite key
try:
    # Use the natural composite key for conflict resolution
    data, count = supabase.table("master_df").upsert(
        records, 
        on_conflict="meeting_key,driver_number"
    ).execute()
    print(f"Successfully upserted {len(records)} records!")
except Exception as e:
    print(f"Error: {e}")

DataFrame shape before deduplication: (651, 21)
Found 0 duplicate rows on ['meeting_key', 'driver_number']
Successfully upserted 651 records!
